# 드론-로봇 연계 배송 — 로봇 배송 용이성 지수 분석

드론이 버티포트에 도착한 이후 라스트마일 배송을 로봇이 담당할 때, 행정동별 로봇 운용 가능성을 정량화합니다.

| 지표 | 출처 | 방향 | 가중치 |
|------|------|------|--------|
| 인도 연결성 지수 | 인도 SHP | 높을수록 유리 | **0.25** |
| 통행 가능 인도 비율 | 인도 SHP | 높을수록 유리 | **0.20** |
| 병목·단절 구간 비율 (역산) | 인도+실폭도로 | 낮을수록 유리 | **0.20** |
| 경사 제약 비율 (역산) | 토지특성정보 | 낮을수록 유리 | **0.15** |
| 맹지 비율 (역산) | 토지특성정보 | 낮을수록 유리 | **0.10** |
| 배송 목적지 밀도 | 토지특성정보 | 높을수록 유리 | **0.10** |

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium import Choropleth, GeoJson
import warnings, glob, os, json

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 로봇 기준값 ─────────────────────────────────────────────
SOLO_MIN      = 1.5   # 단독 통행 가능 최소 폭 (m)
COEXIST_MIN   = 1.0   # 공존 통행 최소 폭 (m)
GAP_M         = 5.0   # 단절 판단 간격 (m)
ROAD_BUF_M    = 5.0   # 도로-인도 인접 버퍼 (m)
HIGHRISE_FL   = 5     # 고층 기준 층수
TARGET_CRS    = 'EPSG:5179'
SEONGNAM_SIG  = ['41131', '41133', '41135']  # 수정·중원·분당구

print('설정 완료')

In [ ]:
NB_DIR       = os.path.dirname(os.path.abspath('__file__'))
BASE_DIR     = os.path.join(NB_DIR, '..')
DATA_DIR     = os.path.join(BASE_DIR, '..', 'Data')
ROBOT_DIR    = os.path.join(DATA_DIR, '로봇용이성')
PROC_DIR     = os.path.join(BASE_DIR, 'processed')

SIDEWALK_PATH = os.path.join(ROBOT_DIR, '인도(보도)데이터', 'N3L_A0033320.shp')
ROAD_PATH     = os.path.join(ROBOT_DIR, '(도로명주소)실폭도로_경기 (1)', 'TL_SPRD_RW_41_202604.shp')
LAND_PATH     = os.path.join(DATA_DIR, '토지특성정보', 'AL_D195_41_20260402.csv')
BOUNDARY_PATH = os.path.join(PROC_DIR, 'seongnam_boundary.gpkg')
MAPPING_PATH  = os.path.join(DATA_DIR, '행정동_법정동_20260325.xlsx')
TOPO_DIRS     = [
    os.path.join(ROBOT_DIR, '수치지형도_SHP_경기_성남시_분당구'),
    os.path.join(ROBOT_DIR, '수치지형도_SHP_경기_성남시_수정구'),
    os.path.join(ROBOT_DIR, '수치지형도_SHP_경기_성남시_중원구'),
]

for label, p in [('인도', SIDEWALK_PATH), ('실폭도로', ROAD_PATH),
                 ('토지', LAND_PATH), ('경계', BOUNDARY_PATH), ('매핑', MAPPING_PATH)]:
    print(f'{label}: {"✓" if os.path.exists(p) else "✗"}')

---
## 기준 레이어 로드 — 행정동 경계

In [ ]:
gdf_bound = gpd.read_file(BOUNDARY_PATH, layer='dong').to_crs(TARGET_CRS)
seongnam_union = gdf_bound.geometry.union_all()  # 성남시 전체 외곽 (클립용)

print(f'행정동 수: {len(gdf_bound)}')
print(f'CRS: {gdf_bound.crs}')
gdf_bound.plot(figsize=(6, 6), edgecolor='k', facecolor='lightyellow')
plt.title('성남시 행정동 경계'); plt.axis('off'); plt.tight_layout(); plt.show()

---
## 1단계 — 인도(보도) SHP 분석

In [ ]:
print('인도 데이터 로드 중...')
gdf_sw = gpd.read_file(SIDEWALK_PATH).to_crs(TARGET_CRS)
print(f'전체 레코드: {len(gdf_sw):,}')
print('컬럼:', gdf_sw.columns.tolist())
print('WIDT 통계:')
print(gdf_sw['WIDT'].describe())

# 성남시 경계 내 클립
gdf_sw = gpd.clip(gdf_sw, seongnam_union)
gdf_sw = gdf_sw[gdf_sw.geometry.is_valid & ~gdf_sw.geometry.is_empty].copy()
print(f'성남시 필터 후: {len(gdf_sw):,}')

In [ ]:
# ── 폭원 3단계 분류 ──────────────────────────────────────────
def classify_width(w):
    if   w >= SOLO_MIN:    return 'solo'      # 단독 통행 가능
    elif w >= COEXIST_MIN: return 'coexist'   # 공존 통행
    else:                  return 'blocked'   # 통행 불가

gdf_sw['width_cls'] = gdf_sw['WIDT'].apply(classify_width)
gdf_sw['length_m']  = gdf_sw.geometry.length

cls_summary = gdf_sw.groupby('width_cls')['length_m'].sum().rename('총 연장(m)')
cls_summary['비율(%)'] = (cls_summary / cls_summary.sum() * 100).round(1)
print('인도 폭원 분류 결과:')
display(cls_summary)

In [ ]:
# ── 시각화 1: 인도 폭원 분류 지도 (matplotlib) ───────────────
COLOR_MAP = {'solo': '#2ca02c', 'coexist': '#ff7f0e', 'blocked': '#d62728'}

fig, ax = plt.subplots(figsize=(10, 10))
gdf_bound.plot(ax=ax, facecolor='#f8f8f8', edgecolor='gray', linewidth=0.5)

for cls, color in COLOR_MAP.items():
    sub = gdf_sw[gdf_sw['width_cls'] == cls]
    if not sub.empty:
        sub.plot(ax=ax, color=color, linewidth=0.4, alpha=0.7)

patches = [
    mpatches.Patch(color='#2ca02c', label=f'단독 통행 가능 (≥{SOLO_MIN}m)'),
    mpatches.Patch(color='#ff7f0e', label=f'공존 통행 ({COEXIST_MIN}~{SOLO_MIN}m)'),
    mpatches.Patch(color='#d62728', label=f'통행 불가 (<{COEXIST_MIN}m)'),
]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_title('성남시 인도 폭원 분류 (로봇 배송 기준)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'robot_sidewalk_width_map.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 인도 연결성 분석 (KDTree 기반) ──────────────────────────────
# LineString 양 끝점 추출
print('연결성 분석 중 (대용량 — 수 분 소요)...')

coords_start = np.array([[g.coords[0][0],  g.coords[0][1]]  for g in gdf_sw.geometry])
coords_end   = np.array([[g.coords[-1][0], g.coords[-1][1]] for g in gdf_sw.geometry])
all_pts = np.vstack([coords_start, coords_end])  # (2N, 2)

tree = cKDTree(all_pts)
n = len(gdf_sw)

# 각 끝점 기준 GAP_M 이내 이웃 존재 여부 → 자신(i, n+i) 제외
nb_start = tree.query_ball_point(coords_start, GAP_M)
nb_end   = tree.query_ball_point(coords_end,   GAP_M)

connected = np.array([
    bool((set(nb_start[i]) - {i, n+i}) or (set(nb_end[i]) - {i, n+i}))
    for i in range(n)
])

gdf_sw['is_connected'] = connected
print(f'연결 구간: {connected.sum():,} / {n:,} ({connected.mean()*100:.1f}%)')

In [ ]:
# ── 행정동별 인도 지표 집계 ──────────────────────────────────────
# 행정동 공간 조인
gdf_sw_j = gpd.sjoin(
    gdf_sw[['geometry','WIDT','width_cls','length_m','is_connected']],
    gdf_bound[['geometry','ADM_NM','GU_NM']],
    how='left', predicate='intersects'
).drop_duplicates(subset=gdf_sw.index.name or gdf_sw.index.names)

sw_stats = gdf_sw_j.groupby('ADM_NM').apply(lambda g: pd.Series({
    'sw_total_len'    : g['length_m'].sum(),
    'sw_passable_len' : g.loc[g['WIDT'] >= COEXIST_MIN, 'length_m'].sum(),
    'sw_solo_len'     : g.loc[g['WIDT'] >= SOLO_MIN,    'length_m'].sum(),
    'sw_total_cnt'    : len(g),
    'sw_connected_cnt': g['is_connected'].sum(),
})).reset_index()

sw_stats['passable_ratio']      = sw_stats['sw_passable_len'] / sw_stats['sw_total_len'].replace(0, np.nan)
sw_stats['connectivity_index']  = sw_stats['sw_connected_cnt'] / sw_stats['sw_total_cnt'].replace(0, np.nan)
sw_stats = sw_stats.fillna(0)

print('인도 지표 (상위 10개 동):')
display(sw_stats.sort_values('connectivity_index', ascending=False).head(10))

---
## 2단계 — 실폭도로 SHP 분석

In [ ]:
print('실폭도로 로드 중...')
gdf_road = gpd.read_file(ROAD_PATH).to_crs(TARGET_CRS)
print(f'전체 레코드: {len(gdf_road):,}')
print('컬럼:', gdf_road.columns.tolist())

# 성남시만 필터
gdf_road = gdf_road[gdf_road['SIG_CD'].isin(SEONGNAM_SIG)].copy()
gdf_road = gdf_road[gdf_road.geometry.is_valid].copy()
print(f'성남시 필터 후: {len(gdf_road):,}')

In [ ]:
# ── 도로 폭원 산출 (area / length * 2 근사) ──────────────────────
# 실폭도로 폴리곤: 면적/둘레*2 ≈ 최소 폭 (세장형 폴리곤에 적합)
gdf_road['road_len_m']   = gdf_road.geometry.length
gdf_road['road_area_m2'] = gdf_road.geometry.area
gdf_road['road_width_m'] = np.where(
    gdf_road['road_len_m'] > 0,
    2 * gdf_road['road_area_m2'] / gdf_road['road_len_m'],
    0
)

print('도로 폭원 통계:')
print(gdf_road['road_width_m'].describe())

# 로봇 이동 가능 도로 (폭 2m 이상, 보도 없어도 최소 통행)
gdf_road['robot_feasible'] = gdf_road['road_width_m'] >= 2.0
print(f'\n로봇 이동 가능 도로: {gdf_road["robot_feasible"].sum():,} / {len(gdf_road):,}')

In [ ]:
# ── 병목 구간: 좁은 인도(WIDT<1.0m)가 도로 인접 구간에 위치 ────────
print('병목 구간 분석 중...')

# 도로 폴리곤 버퍼 (ROAD_BUF_M)
gdf_road_buf = gdf_road[['geometry','SIG_CD']].copy()
gdf_road_buf.geometry = gdf_road_buf.geometry.buffer(ROAD_BUF_M)

# 좁은 인도 추출
narrow_sw = gdf_sw_j[gdf_sw_j['WIDT'] < COEXIST_MIN][['geometry','ADM_NM','length_m']].copy()

# 병목 = 좁은 인도 ∩ 도로 버퍼
bottleneck = gpd.sjoin(
    narrow_sw, gdf_road_buf[['geometry']], how='inner', predicate='intersects'
).drop_duplicates(subset=['geometry'])

print(f'병목 구간 수: {len(bottleneck):,}')

In [ ]:
# ── 단절 구간: 도로 인근에 인도가 없는 구간 ─────────────────────────
print('단절 구간 분석 중...')

# 도로 폴리곤 중심점 기준으로 인근 인도 존재 여부 확인
road_centroids = gdf_road.copy()
road_centroids.geometry = gdf_road.geometry.centroid

# sjoin_nearest: 각 도로 중심에서 가장 가까운 인도까지 거리
road_sw_nearest = gpd.sjoin_nearest(
    road_centroids[['geometry','SIG_CD','road_len_m']],
    gdf_sw[['geometry']],
    how='left',
    max_distance=ROAD_BUF_M,
    distance_col='dist_to_sw'
)
road_sw_nearest = road_sw_nearest[~road_sw_nearest.index.duplicated()]

# 인도 없는 도로 = 단절 구간
no_sw_roads = road_sw_nearest[road_sw_nearest['dist_to_sw'].isna()]
print(f'인도 없는 도로 구간: {len(no_sw_roads):,} / {len(gdf_road):,}')

In [ ]:
# ── 행정동별 도로 지표 집계 ──────────────────────────────────────────
# 도로 중심점 → 행정동 조인
road_dong = gpd.sjoin(
    road_sw_nearest[['geometry','road_len_m','dist_to_sw']],
    gdf_bound[['geometry','ADM_NM']],
    how='left', predicate='within'
).drop_duplicates()

road_stats = road_dong.groupby('ADM_NM').apply(lambda g: pd.Series({
    'road_total_cnt'   : len(g),
    'road_no_sw_cnt'   : g['dist_to_sw'].isna().sum(),
})).reset_index()
road_stats['disconnect_ratio'] = road_stats['road_no_sw_cnt'] / road_stats['road_total_cnt'].replace(0, np.nan)

# 병목 구간 행정동별 집계
bottleneck_stats = bottleneck.groupby('ADM_NM').agg(
    bottleneck_cnt=('length_m', 'count'),
    bottleneck_len=('length_m', 'sum')
).reset_index()

road_stats = road_stats.merge(bottleneck_stats, on='ADM_NM', how='left').fillna(0)
road_stats['bottleneck_per_km'] = road_stats['bottleneck_cnt'] / (road_stats['road_total_cnt'] / 1000 + 1e-6)

print('도로 지표 (상위 10개 동):')
display(road_stats.sort_values('disconnect_ratio', ascending=False).head(10))

---
## 3단계 — 수치지형도 건물 SHP 분석 (고층 건물 비율)

In [ ]:
# N3A_B0010000.shp — 건물 레이어 (층수 포함)
print('수치지형도 건물 레이어 로드 중...')
bldg_files = []
for topo_dir in TOPO_DIRS:
    bldg_files += glob.glob(os.path.join(topo_dir, '**', 'N3A_B0010000.shp'), recursive=True)

print(f'건물 SHP 파일 수: {len(bldg_files)}')

frames = []
for fp in bldg_files:
    try:
        g = gpd.read_file(fp)
        if g.empty: continue
        frames.append(g[['geometry'] + [c for c in g.columns if '층' in c or '수' in c or 'UFID' in c]])
    except Exception:
        continue

if frames:
    gdf_bldg = pd.concat(frames, ignore_index=True)
    gdf_bldg = gpd.GeoDataFrame(gdf_bldg, crs=frames[0].crs).to_crs(TARGET_CRS)
    print(f'건물 총 레코드: {len(gdf_bldg):,}')
    print('컬럼:', gdf_bldg.columns.tolist())
else:
    print('건물 레이어 없음')
    gdf_bldg = gpd.GeoDataFrame()

In [ ]:
# 층수 컬럼 자동 탐색 및 고층 비율 산출
if not gdf_bldg.empty:
    floor_col = next((c for c in gdf_bldg.columns if '층수' in c or '층' in c), None)
    print(f'층수 컬럼: {floor_col}')

    if floor_col:
        gdf_bldg[floor_col] = pd.to_numeric(gdf_bldg[floor_col], errors='coerce')
        gdf_bldg['is_highrise'] = gdf_bldg[floor_col] >= HIGHRISE_FL

        # 행정동 조인 (건물 중심점)
        bldg_c = gdf_bldg.copy()
        bldg_c.geometry = bldg_c.geometry.centroid
        bldg_dong = gpd.sjoin(
            bldg_c[['geometry','is_highrise']],
            gdf_bound[['geometry','ADM_NM']],
            how='left', predicate='within'
        ).drop_duplicates()

        bldg_stats = bldg_dong.groupby('ADM_NM').apply(lambda g: pd.Series({
            'bldg_total'   : len(g),
            'bldg_highrise': g['is_highrise'].sum(),
        })).reset_index()
        bldg_stats['highrise_ratio'] = bldg_stats['bldg_highrise'] / bldg_stats['bldg_total'].replace(0, np.nan)
        bldg_stats = bldg_stats.fillna(0)
        print('고층 건물 비율 (상위 10):')
        display(bldg_stats.sort_values('highrise_ratio', ascending=False).head(10))
    else:
        bldg_stats = pd.DataFrame(columns=['ADM_NM','highrise_ratio'])
else:
    bldg_stats = pd.DataFrame(columns=['ADM_NM','highrise_ratio'])

---
## 4단계 — 토지특성정보 CSV 분석

In [ ]:
# 컬럼 확인용 샘플 로드
df_sample = pd.read_csv(LAND_PATH, encoding='cp949', nrows=3)
print('토지특성정보 컬럼:')
print(df_sample.columns.tolist())

In [ ]:
# 필요 컬럼만 로드 (메모리 절약)
USE_COLS = ['법정동코드', '법정동명', '지형높이코드', '지형높이', 
            '도로접면코드', '도로접면', '토지이용상황코드', '토지이용상황',
            '용도지역코드1', '용도지역명1', '토지면적']
# 실제 존재하는 컬럼만 선택
actual_cols = [c for c in USE_COLS if c in df_sample.columns]

print('성남시 필터로 로드 중 (대용량 — 수 분 소요)...')
chunks = pd.read_csv(LAND_PATH, encoding='cp949', usecols=actual_cols,
                     dtype={'법정동코드': str}, chunksize=200_000)

df_land_list = []
for chunk in chunks:
    # 성남시 법정동코드: 4113으로 시작 (10자리)
    sn = chunk[chunk['법정동코드'].str.startswith('4113', na=False)]
    if not sn.empty:
        df_land_list.append(sn)

df_land = pd.concat(df_land_list, ignore_index=True)
df_land['법정동코드'] = df_land['법정동코드'].astype(int)
print(f'성남시 토지 필지 수: {len(df_land):,}')
display(df_land.head(3))

In [ ]:
# 고유값 탐색 (코드 확인)
print('지형높이 고유값:',  df_land['지형높이'].value_counts().head(10).to_dict())
print('도로접면 고유값:',  df_land['도로접면'].value_counts().head(10).to_dict())
print('토지이용상황 고유값:', df_land['토지이용상황'].value_counts().head(15).to_dict())

In [ ]:
# ── 법정동 → 행정동 매핑 ─────────────────────────────────────────
df_map = pd.read_excel(MAPPING_PATH)
df_map_sn = df_map[
    df_map['말소일자'].isna() &
    df_map['시군구명'].str.contains('성남', na=False) &
    df_map['읍면동명'].notna()
][['법정동코드','읍면동명']].drop_duplicates()
df_map_sn.columns = ['법정동코드','ADM_NM']
df_map_sn['법정동코드'] = df_map_sn['법정동코드'].astype(int)

# 1:N 매핑 → 각 필지가 해당하는 모든 행정동에 배분
df_land_dong = df_land.merge(df_map_sn, on='법정동코드', how='left')
df_land_dong['ADM_NM'] = df_land_dong['ADM_NM'].fillna('미매핑')
print(f'매핑 성공: {(df_land_dong["ADM_NM"] != "미매핑").sum():,} / {len(df_land_dong):,}')

In [ ]:
# ── 행정동별 토지 지표 산출 ──────────────────────────────────────
SLOPE_KEYWORDS   = ['급경사', '고지']
BLIND_KEYWORDS   = ['맹지']
DELIVERY_KEYWORDS = ['단독주택', '연립주택', '다세대', '아파트', '상업용', '주거용',
                     '상업나지', '주거나지', '연립다세대']

df_l = df_land_dong[df_land_dong['ADM_NM'] != '미매핑'].copy()

df_l['is_slope']    = df_l['지형높이'].apply(lambda x: any(k in str(x) for k in SLOPE_KEYWORDS))
df_l['is_blind']    = df_l['도로접면'].apply(lambda x: any(k in str(x) for k in BLIND_KEYWORDS))
df_l['is_delivery'] = df_l['토지이용상황'].apply(lambda x: any(k in str(x) for k in DELIVERY_KEYWORDS))

land_stats = df_l.groupby('ADM_NM').apply(lambda g: pd.Series({
    'parcel_total'   : len(g),
    'slope_cnt'      : g['is_slope'].sum(),
    'blind_cnt'      : g['is_blind'].sum(),
    'delivery_cnt'   : g['is_delivery'].sum(),
})).reset_index()

land_stats['slope_ratio']    = land_stats['slope_cnt']    / land_stats['parcel_total'].replace(0, np.nan)
land_stats['blind_ratio']    = land_stats['blind_cnt']    / land_stats['parcel_total'].replace(0, np.nan)
land_stats['delivery_density'] = land_stats['delivery_cnt'] / land_stats['parcel_total'].replace(0, np.nan)
land_stats = land_stats.fillna(0)

print('토지 지표 상위 10동:')
display(land_stats.sort_values('delivery_density', ascending=False).head(10))

---
## 5단계 — 로봇 배송 용이성 지수 통합 산출

In [ ]:
def minmax(s):
    mn, mx = s.min(), s.max()
    if mx == mn: return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

# 전체 행정동 기준 테이블
base = gdf_bound[['ADM_NM','GU_NM']].copy()

df_idx = (
    base
    .merge(sw_stats[['ADM_NM','passable_ratio','connectivity_index']], on='ADM_NM', how='left')
    .merge(road_stats[['ADM_NM','disconnect_ratio','bottleneck_cnt']], on='ADM_NM', how='left')
    .merge(land_stats[['ADM_NM','slope_ratio','blind_ratio','delivery_density']], on='ADM_NM', how='left')
)

if not bldg_stats.empty and 'highrise_ratio' in bldg_stats.columns:
    df_idx = df_idx.merge(bldg_stats[['ADM_NM','highrise_ratio']], on='ADM_NM', how='left')
else:
    df_idx['highrise_ratio'] = 0.0

df_idx = df_idx.fillna(0)

# Min-Max 정규화
df_idx['n_passable']      = minmax(df_idx['passable_ratio'])
df_idx['n_connectivity']  = minmax(df_idx['connectivity_index'])
df_idx['n_bottleneck']    = 1 - minmax(df_idx['bottleneck_cnt'])      # 역산
df_idx['n_disconnect']    = 1 - minmax(df_idx['disconnect_ratio'])    # 역산
df_idx['n_slope']         = 1 - minmax(df_idx['slope_ratio'])         # 역산
df_idx['n_blind']         = 1 - minmax(df_idx['blind_ratio'])         # 역산
df_idx['n_delivery']      = minmax(df_idx['delivery_density'])

# 병목+단절 결합 (평균)
df_idx['n_obstacle_combined'] = (df_idx['n_bottleneck'] + df_idx['n_disconnect']) / 2

# 가중 합산
df_idx['robot_index'] = (
    df_idx['n_connectivity']       * 0.25 +
    df_idx['n_passable']           * 0.20 +
    df_idx['n_obstacle_combined']  * 0.20 +
    df_idx['n_slope']              * 0.15 +
    df_idx['n_blind']              * 0.10 +
    df_idx['n_delivery']           * 0.10
)
df_idx['robot_index_100'] = (df_idx['robot_index'] * 100).round(2)

result = df_idx[['ADM_NM','GU_NM','robot_index_100',
                  'connectivity_index','passable_ratio',
                  'slope_ratio','blind_ratio','delivery_density']].sort_values('robot_index_100', ascending=False)

print('=== 로봇 배송 용이성 지수 TOP 20 ===')
display(result.head(20))

---
## 6단계 — 시각화
### 6-1. 행정동별 용이성 지수 막대 차트

In [ ]:
fig_bar = px.bar(
    result.sort_values('robot_index_100'),
    x='robot_index_100', y='ADM_NM',
    orientation='h',
    color='GU_NM',
    color_discrete_map={'수정구':'#e63946','중원구':'#457b9d','분당구':'#2a9d8f'},
    text='robot_index_100',
    title='행정동별 로봇 배송 용이성 지수 (0~100)',
    labels={'robot_index_100':'용이성 지수', 'ADM_NM':'행정동', 'GU_NM':'구'}
)
fig_bar.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig_bar.update_layout(height=900, plot_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.01))
fig_bar.show()

### 6-2. 구별 용이성 지수 분포 박스플롯

In [ ]:
fig_box = px.box(
    df_idx, x='GU_NM', y='robot_index_100',
    color='GU_NM',
    color_discrete_map={'수정구':'#e63946','중원구':'#457b9d','분당구':'#2a9d8f'},
    points='all',
    hover_data=['ADM_NM'],
    title='구별 로봇 배송 용이성 지수 분포',
    labels={'GU_NM':'구', 'robot_index_100':'용이성 지수 (0~100)'}
)
fig_box.update_layout(height=500, plot_bgcolor='white', showlegend=False)
fig_box.show()

print('구별 평균:')
display(df_idx.groupby('GU_NM')['robot_index_100'].describe().round(2))

### 6-3. 단계구분도 (Choropleth Map)

In [ ]:
# 경계 + 지수 합류
gdf_result = gdf_bound.to_crs('EPSG:4326').merge(
    df_idx[['ADM_NM','GU_NM','robot_index_100',
             'connectivity_index','passable_ratio','slope_ratio','blind_ratio']],
    on='ADM_NM', how='left'
)

geojson = json.loads(gdf_result.to_json())
center  = [gdf_result.geometry.centroid.y.mean(), gdf_result.geometry.centroid.x.mean()]

m = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')

Choropleth(
    geo_data=geojson,
    data=gdf_result.dropna(subset=['robot_index_100']),
    columns=['ADM_NM','robot_index_100'],
    key_on='feature.properties.ADM_NM',
    fill_color='RdYlGn',
    fill_opacity=0.75,
    line_opacity=0.5,
    legend_name='로봇 배송 용이성 지수 (0~100)',
    nan_fill_color='#cccccc'
).add_to(m)

GeoJson(
    geojson,
    tooltip=folium.GeoJsonTooltip(
        fields=['ADM_NM','GU_NM','robot_index_100','connectivity_index','passable_ratio','slope_ratio'],
        aliases=['행정동','구','용이성 지수','인도 연결성','통행 가능 비율','경사 제약 비율'],
        localize=True, sticky=True
    ),
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0}
).add_to(m)

# 상위 10개 동 마커
for _, row in gdf_result.sort_values('robot_index_100', ascending=False).head(10).iterrows():
    c = row.geometry.centroid
    folium.Marker(
        [c.y, c.x],
        popup=folium.Popup(
            f"<b>{row['ADM_NM']}</b><br>용이성: {row['robot_index_100']:.1f}<br>"
            f"인도 연결성: {row['connectivity_index']:.2f}<br>"
            f"경사 제약: {row['slope_ratio']:.2f}",
            max_width=200),
        icon=folium.Icon(color='green', icon='robot', prefix='fa')
    ).add_to(m)

map_path = os.path.join(PROC_DIR, 'robot_index_map.html')
m.save(map_path)
print(f'지도 저장: {map_path}')
m

### 6-4. 인도 네트워크 오버레이 (통행 가능·병목·단절 색상 코딩)

In [ ]:
# 단절 인도 = is_connected=False
gdf_sw_j['network_cls'] = np.where(
    ~gdf_sw_j['is_connected'], 'disconnected',
    np.where(gdf_sw_j['WIDT'] < COEXIST_MIN, 'bottleneck', 'passable')
)

NET_COLOR = {'passable':'#2ca02c', 'bottleneck':'#ff7f0e', 'disconnected':'#d62728'}

fig2, ax2 = plt.subplots(figsize=(12, 12))
gdf_bound.plot(ax=ax2, facecolor='#f5f5f5', edgecolor='#888', linewidth=0.8, zorder=1)

for cls, color in NET_COLOR.items():
    sub = gdf_sw_j[gdf_sw_j['network_cls'] == cls]
    if not sub.empty:
        sub.plot(ax=ax2, color=color, linewidth=0.5, alpha=0.8, zorder=2)

patches2 = [
    mpatches.Patch(color='#2ca02c', label='통행 가능 (≥1.0m + 연결)'),
    mpatches.Patch(color='#ff7f0e', label='병목 구간 (<1.0m)'),
    mpatches.Patch(color='#d62728', label='단절 구간 (연결 끊김)'),
]
ax2.legend(handles=patches2, loc='lower right', fontsize=10)
ax2.set_title('성남시 인도 네트워크 — 로봇 통행 적합성', fontsize=14)
ax2.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(PROC_DIR, 'robot_sidewalk_network.png'), dpi=150, bbox_inches='tight')
plt.show()

### 6-5. 지표 레이더 차트 (상위 구별 비교)

In [ ]:
radar_cols = ['n_connectivity','n_passable','n_obstacle_combined',
              'n_slope','n_blind','n_delivery']
radar_labels = ['인도 연결성','통행 가능 비율','병목·단절(역산)',
                '경사 제약(역산)','맹지(역산)','배송 목적지']

gu_radar = df_idx.groupby('GU_NM')[radar_cols].mean().reset_index()

fig_radar = go.Figure()
colors_r = {'수정구':'#e63946','중원구':'#457b9d','분당구':'#2a9d8f'}

for _, row in gu_radar.iterrows():
    vals = row[radar_cols].tolist()
    vals += [vals[0]]  # 닫기
    fig_radar.add_trace(go.Scatterpolar(
        r=vals, theta=radar_labels + [radar_labels[0]],
        fill='toself', name=row['GU_NM'],
        line_color=colors_r.get(row['GU_NM'],'gray'),
        opacity=0.6
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    title='구별 로봇 배송 용이성 지표 레이더 차트',
    height=500
)
fig_radar.show()

---
## 결과 저장

In [ ]:
# CSV 저장
out_path = os.path.join(PROC_DIR, 'robot_delivery_index.csv')
result.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장: {out_path}')

print('\n=== 최종 요약 ===')
print(f'분석 행정동 수: {len(result)}개')
print(f'\n[상위 5개 — 즉시 로봇 배송 투입 권장]')
for _, r in result.head(5).iterrows():
    print(f'  {r["ADM_NM"]:8s} ({r["GU_NM"]}) | 지수 {r["robot_index_100"]:5.1f}')

print(f'\n[하위 5개 — 인프라 개선 필요]')
for _, r in result.tail(5).iterrows():
    print(f'  {r["ADM_NM"]:8s} ({r["GU_NM"]}) | 지수 {r["robot_index_100"]:5.1f}')